# Threat Exposure Management — Interactive Dashboard

This notebook builds an interactive dashboard from your PowerPoint deck.

**Before running:** update `PPTX_PATH` in the next cell to your local file path:
```
C:\\Users\\GupteP\\Downloads\\Threat_Exposure_Management_July_2026.pptx
```

In [ ]:
# Install dependencies (run once)
# !pip install -r requirements.txt

In [ ]:
import re
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from pptx import Presentation

# --- CONFIG: set your PowerPoint path here ---
PPTX_PATH = r"C:\Users\GupteP\Downloads\Threat_Exposure_Management_July_2026.pptx"

# Optional: export extracted tables to CSV for manual cleanup
EXPORT_DIR = Path("./extracted_data")
EXPORT_DIR.mkdir(exist_ok=True)

## 1. Extract content from PowerPoint

In [ ]:
def extract_text_from_shape(shape):
    """Pull all text from a shape, including table cells."""
    texts = []
    if shape.has_text_frame:
        for paragraph in shape.text_frame.paragraphs:
            line = "".join(run.text for run in paragraph.runs).strip()
            if line:
                texts.append(line)
    if shape.has_table:
        for row in shape.table.rows:
            row_vals = [cell.text.strip() for cell in row.cells]
            if any(row_vals):
                texts.append(" | ".join(row_vals))
    return texts


def table_shape_to_dataframe(shape):
    """Convert a PPT table shape into a pandas DataFrame."""
    if not shape.has_table:
        return None
    rows = []
    for row in shape.table.rows:
        rows.append([cell.text.strip() for cell in row.cells])
    if not rows:
        return None
    header, *body = rows
    if body:
        return pd.DataFrame(body, columns=header)
    return pd.DataFrame([header])


def parse_pptx(path: str) -> dict:
    """Extract slide titles, bullet text, and tables from a .pptx file."""
    prs = Presentation(path)
    slides = []
    all_tables = []

    for idx, slide in enumerate(prs.slides, start=1):
        slide_title = ""
        bullets = []
        tables = []

        for shape in slide.shapes:
            if shape.has_text_frame and not slide_title:
                # First text box is often the title
                candidate = shape.text_frame.text.strip()
                if candidate:
                    slide_title = candidate.split("\n")[0]

            bullets.extend(extract_text_from_shape(shape))

            df = table_shape_to_dataframe(shape)
            if df is not None and not df.empty:
                df.insert(0, "slide", idx)
                df.insert(1, "slide_title", slide_title)
                tables.append(df)
                all_tables.append(df)

        slides.append({
            "slide": idx,
            "title": slide_title or f"Slide {idx}",
            "bullets": bullets,
        })

    slides_df = pd.DataFrame(slides)
    tables_df = pd.concat(all_tables, ignore_index=True) if all_tables else pd.DataFrame()
    return {"slides": slides_df, "tables": tables_df}


ppt_data = parse_pptx(PPTX_PATH)
slides_df = ppt_data["slides"]
tables_df = ppt_data["tables"]

print(f"Loaded {len(slides_df)} slides")
print(f"Extracted {len(tables_df)} table rows from PPT")
display(slides_df.head())
if not tables_df.empty:
    display(tables_df.head())
    tables_df.to_csv(EXPORT_DIR / "ppt_tables_raw.csv", index=False)
    print(f"Saved raw tables to {EXPORT_DIR / 'ppt_tables_raw.csv'}")

## 2. Build dashboard datasets

If your PPT tables map cleanly to exposure metrics, the notebook uses them directly.
Otherwise, sample data is generated so you can preview the dashboard layout.

In [ ]:
def _looks_numeric(series: pd.Series) -> bool:
    converted = pd.to_numeric(series.astype(str).str.replace(",", ""), errors="coerce")
    return converted.notna().mean() > 0.6


def build_exposure_dataframe(tables: pd.DataFrame) -> pd.DataFrame:
    """Try to normalize PPT tables into a standard exposure schema."""
    if tables.empty:
        return pd.DataFrame()

    # Heuristic: find columns that look like asset / severity / score fields
    cols = {c.lower(): c for c in tables.columns if c not in ("slide", "slide_title")}

    asset_col = next((cols[k] for k in cols if re.search(r"asset|host|system|name", k)), None)
    severity_col = next((cols[k] for k in cols if re.search(r"severity|risk|priority", k)), None)
    score_col = next((cols[k] for k in cols if re.search(r"score|exposure|rating", k)), None)
    status_col = next((cols[k] for k in cols if re.search(r"status|state|remediation", k)), None)

    if asset_col is None:
        return pd.DataFrame()

    out = pd.DataFrame({
        "asset": tables[asset_col].astype(str),
        "severity": tables[severity_col].astype(str) if severity_col else "Unknown",
        "exposure_score": pd.to_numeric(
            tables[score_col].astype(str).str.replace(",", ""), errors="coerce"
        ) if score_col else np.nan,
        "status": tables[status_col].astype(str) if status_col else "Open",
        "source_slide": tables["slide"],
    })
    out["exposure_score"] = out["exposure_score"].fillna(50)
    return out.drop_duplicates().reset_index(drop=True)


def sample_exposure_data(n: int = 120) -> pd.DataFrame:
    """Fallback sample data for Threat Exposure Management dashboards."""
    rng = np.random.default_rng(42)
    assets = [f"SRV-{i:03d}" for i in range(1, n + 1)]
    severities = rng.choice(["Critical", "High", "Medium", "Low"], n, p=[0.08, 0.22, 0.40, 0.30])
    statuses = rng.choice(["Open", "In Progress", "Remediated", "Accepted"], n, p=[0.35, 0.25, 0.30, 0.10])
    categories = rng.choice(
        ["Vulnerability", "Misconfiguration", "Identity", "Data Exposure", "Third Party"],
        n,
    )
    scores = np.clip(rng.normal(62, 18, n), 5, 100).round(1)
    days_open = rng.integers(1, 180, n)

    return pd.DataFrame({
        "asset": assets,
        "severity": severities,
        "category": categories,
        "exposure_score": scores,
        "status": statuses,
        "days_open": days_open,
        "last_seen": [datetime.today() - timedelta(days=int(d)) for d in days_open],
    })


exposure_df = build_exposure_dataframe(tables_df)
using_sample = exposure_df.empty

if using_sample:
    exposure_df = sample_exposure_data()
    print("Using sample data — replace with your PPT table mapping once column names are confirmed.")
else:
    print("Using data extracted from PowerPoint tables.")

if "category" not in exposure_df.columns:
    exposure_df["category"] = "General"
if "days_open" not in exposure_df.columns:
    exposure_df["days_open"] = np.random.default_rng(7).integers(1, 120, len(exposure_df))
if "last_seen" not in exposure_df.columns:
    exposure_df["last_seen"] = datetime.today() - pd.to_timedelta(exposure_df["days_open"], unit="D")

exposure_df.to_csv(EXPORT_DIR / "exposure_data.csv", index=False)
exposure_df.head()

## 3. KPI helpers and chart builders

In [ ]:
SEVERITY_ORDER = ["Critical", "High", "Medium", "Low", "Unknown"]
SEVERITY_COLORS = {
    "Critical": "#B42318",
    "High": "#F04438",
    "Medium": "#F79009",
    "Low": "#12B76A",
    "Unknown": "#98A2B3",
}


def compute_kpis(df: pd.DataFrame) -> dict:
    open_mask = df["status"].isin(["Open", "In Progress"])
    return {
        "total_assets": df["asset"].nunique(),
        "open_findings": int(open_mask.sum()),
        "critical_open": int((open_mask & (df["severity"] == "Critical")).sum()),
        "avg_exposure": round(df.loc[open_mask, "exposure_score"].mean(), 1) if open_mask.any() else 0,
        "remediated_pct": round(100 * (df["status"] == "Remediated").mean(), 1),
    }


def kpi_cards_html(kpis: dict) -> str:
    cards = [
        ("Total Assets", kpis["total_assets"]),
        ("Open Findings", kpis["open_findings"]),
        ("Critical Open", kpis["critical_open"]),
        ("Avg Exposure Score", kpis["avg_exposure"]),
        ("Remediated %", f"{kpis['remediated_pct']}%"),
    ]
    html = '<div style="display:flex; gap:12px; flex-wrap:wrap; margin-bottom:16px;">'
    for label, value in cards:
        html += f'''
        <div style="flex:1; min-width:150px; background:#101828; color:#fff; border-radius:10px; padding:14px 16px;">
            <div style="font-size:12px; opacity:0.75;">{label}</div>
            <div style="font-size:26px; font-weight:700; margin-top:4px;">{value}</div>
        </div>'''
    html += "</div>"
    return html


def chart_severity_distribution(df: pd.DataFrame):
    counts = (
        df["severity"]
        .value_counts()
        .reindex([s for s in SEVERITY_ORDER if s in df["severity"].unique()])
        .dropna()
        .reset_index()
    )
    counts.columns = ["severity", "count"]
    fig = px.bar(
        counts,
        x="severity",
        y="count",
        color="severity",
        color_discrete_map=SEVERITY_COLORS,
        title="Findings by Severity",
    )
    fig.update_layout(showlegend=False, template="plotly_white", height=360)
    return fig


def chart_exposure_by_category(df: pd.DataFrame):
    agg = df.groupby("category", as_index=False)["exposure_score"].mean().sort_values("exposure_score", ascending=False)
    fig = px.bar(
        agg,
        x="category",
        y="exposure_score",
        title="Average Exposure Score by Category",
        color="exposure_score",
        color_continuous_scale="Reds",
    )
    fig.update_layout(template="plotly_white", height=360, coloraxis_showscale=False)
    return fig


def chart_status_breakdown(df: pd.DataFrame):
    status_counts = df["status"].value_counts().reset_index()
    status_counts.columns = ["status", "count"]
    fig = px.pie(status_counts, names="status", values="count", hole=0.45, title="Remediation Status")
    fig.update_layout(template="plotly_white", height=360)
    return fig


def chart_top_risk_assets(df: pd.DataFrame, top_n: int = 15):
    top = df.nlargest(top_n, "exposure_score")[["asset", "exposure_score", "severity", "status"]]
    fig = px.bar(
        top,
        x="exposure_score",
        y="asset",
        orientation="h",
        color="severity",
        color_discrete_map=SEVERITY_COLORS,
        title=f"Top {top_n} Assets by Exposure Score",
    )
    fig.update_layout(template="plotly_white", height=420, yaxis={"categoryorder": "total ascending"})
    return fig


def chart_exposure_trend(df: pd.DataFrame):
  # Synthetic weekly trend from current snapshot when historical data is unavailable
    base = df["exposure_score"].mean()
    weeks = pd.date_range(end=datetime.today(), periods=8, freq="W")
    trend = base + np.linspace(8, -4, len(weeks)) + np.random.default_rng(1).normal(0, 1.5, len(weeks))
    trend_df = pd.DataFrame({"week": weeks, "avg_exposure_score": np.clip(trend, 0, 100)})
    fig = px.line(trend_df, x="week", y="avg_exposure_score", markers=True, title="Exposure Score Trend (8 weeks)")
    fig.update_layout(template="plotly_white", height=360, yaxis_range=[0, 100])
    return fig

## 4. Interactive dashboard

In [ ]:
severity_options = sorted(exposure_df["severity"].dropna().unique(), key=lambda s: SEVERITY_ORDER.index(s) if s in SEVERITY_ORDER else 99)
category_options = sorted(exposure_df["category"].dropna().unique())
status_options = sorted(exposure_df["status"].dropna().unique())

severity_filter = widgets.SelectMultiple(
    options=severity_options,
    value=tuple(severity_options),
    description="Severity",
    layout=widgets.Layout(width="220px", height="120px"),
)
category_filter = widgets.SelectMultiple(
    options=category_options,
    value=tuple(category_options),
    description="Category",
    layout=widgets.Layout(width="220px", height="120px"),
)
status_filter = widgets.SelectMultiple(
    options=status_options,
    value=tuple(status_options),
    description="Status",
    layout=widgets.Layout(width="220px", height="120px"),
)
min_score = widgets.FloatSlider(
    value=0,
    min=0,
    max=100,
    step=1,
    description="Min score",
    continuous_update=False,
    layout=widgets.Layout(width="320px"),
)
search_box = widgets.Text(
    value="",
    placeholder="Search asset...",
    description="Search",
    layout=widgets.Layout(width="320px"),
)
update_btn = widgets.Button(description="Update Dashboard", button_style="primary", icon="refresh")
export_btn = widgets.Button(description="Export Filtered CSV", icon="download")

dashboard_out = widgets.Output()
table_out = widgets.Output()


def apply_filters(df: pd.DataFrame) -> pd.DataFrame:
    filtered = df.copy()
    if severity_filter.value:
        filtered = filtered[filtered["severity"].isin(severity_filter.value)]
    if category_filter.value:
        filtered = filtered[filtered["category"].isin(category_filter.value)]
    if status_filter.value:
        filtered = filtered[filtered["status"].isin(status_filter.value)]
    filtered = filtered[filtered["exposure_score"] >= min_score.value]
    if search_box.value.strip():
        filtered = filtered[filtered["asset"].str.contains(search_box.value.strip(), case=False, na=False)]
    return filtered


def render_dashboard(_=None):
    with dashboard_out:
        clear_output(wait=True)
        filtered = apply_filters(exposure_df)
        kpis = compute_kpis(filtered)

        display(HTML(f"<h2 style='margin:0 0 8px 0;'>Threat Exposure Management — July 2026</h2>"))
        display(HTML(kpi_cards_html(kpis)))

        row1 = widgets.HBox([
            widgets.Output(),
            widgets.Output(),
            widgets.Output(),
        ])
        with row1.children[0]:
            display(chart_severity_distribution(filtered))
        with row1.children[1]:
            display(chart_status_breakdown(filtered))
        with row1.children[2]:
            display(chart_exposure_by_category(filtered))
        display(row1)

        row2 = widgets.HBox([widgets.Output(), widgets.Output()])
        with row2.children[0]:
            display(chart_top_risk_assets(filtered))
        with row2.children[1]:
            display(chart_exposure_trend(filtered))
        display(row2)

    with table_out:
        clear_output(wait=True)
        display(HTML("<h3>Filtered findings</h3>"))
        display(apply_filters(exposure_df).sort_values("exposure_score", ascending=False).head(50))


def export_filtered(_=None):
    filtered = apply_filters(exposure_df)
    out_path = EXPORT_DIR / "filtered_exposure_export.csv"
    filtered.to_csv(out_path, index=False)
    print(f"Exported {len(filtered)} rows to {out_path}")


update_btn.on_click(render_dashboard)
export_btn.on_click(export_filtered)

controls = widgets.VBox([
    widgets.HTML("<b>Filters</b>"),
    widgets.HBox([severity_filter, category_filter, status_filter]),
    widgets.HBox([min_score, search_box]),
    widgets.HBox([update_btn, export_btn]),
])

display(controls)
display(dashboard_out)
display(table_out)
render_dashboard()

## 5. Optional — browse slide content from your PPT

In [ ]:
slide_picker = widgets.Dropdown(
    options=[(row["title"], row["slide"]) for _, row in slides_df.iterrows()],
    description="Slide",
    layout=widgets.Layout(width="500px"),
)
slide_out = widgets.Output()


def show_slide_content(change=None):
    with slide_out:
        clear_output(wait=True)
        slide_no = slide_picker.value
        row = slides_df.loc[slides_df["slide"] == slide_no].iloc[0]
        display(HTML(f"<h3>Slide {slide_no}: {row['title']}</h3>"))
        if row["bullets"]:
            display(HTML("<ul>" + "".join(f"<li>{b}</li>" for b in row["bullets"]) + "</ul>"))
        slide_tables = tables_df[tables_df["slide"] == slide_no] if not tables_df.empty else pd.DataFrame()
        if not slide_tables.empty:
            display(slide_tables.drop(columns=["slide", "slide_title"], errors="ignore"))


slide_picker.observe(show_slide_content, names="value")
display(slide_picker)
display(slide_out)
show_slide_content()